# NACC_DemRKL
Copied from the original project. Paths are relative to the main folder.
Original DemRKG generation and training settings are preserved.


In [ ]:
from pathlib import Path
import os
import sys
_starts = [Path.cwd(), Path(globals().get("__file__", Path.cwd() / "notebook")).resolve().parent]
CLINICALKG_ROOT = next(p for start in _starts for p in (start, *start.parents)
                      if (p / "run" / "common" / "paths.py").is_file())
sys.path.insert(0, str(CLINICALKG_ROOT.parent))
from clinicalKG.run.common.paths import PROJECT_ROOT, SHARED_PRIMEKG, ANALYSIS_ROOT, raw_primekg_path, work_dir, asset_path, dataset_dir, result_path
DATASET = 'NACC'
METHOD = 'KAMT_DemRKG'
WORK_DIR = work_dir(DATASET, METHOD) if METHOD != "CustomKG" else CLINICALKG_ROOT / "run" / "common"
DATASET_DIR = dataset_dir(DATASET)
os.environ.setdefault("KG_ABLATION_OUTPUT", str(ANALYSIS_ROOT))
if DATASET_DIR.is_dir():
    os.chdir(DATASET_DIR)


In [ ]:
import pandas as pd
import warnings

warnings.filterwarnings('ignore')

RAW_KG_PATH = str(raw_primekg_path())

def precise_search_missed_nodes():
    print("==================================================")
    print("🎯 启动未命中节点精准检索...")
    
    try:
        df = pd.read_csv(RAW_KG_PATH, low_memory=False)
    except FileNotFoundError:
        print("❌ 找不到 kg.csv，请检查路径。")
        return

    target_types = ['disease', 'effect/phenotype']
    search_df = df[df['x_type'].isin(target_types)]
    
    missed_terms = {
        "房颤 (AFIB)": "atrial fibrillation",
        "短暂性脑缺血 (TIA)": "transient ischemic attack",
        "脑外伤 (TBI)": "traumatic brain",
        "高胆固醇 (HYPERCHO)": "hypercholesterolemia",
        "糖尿病 (DIABETES)": "diabetes"
    }
    
    for name, keyword in missed_terms.items():
        print(f"\n▶ 正在精准检索【{name}】...")
        
        mask = search_df['x_name'].str.contains(keyword, case=False, na=False)
        results = search_df[mask]['x_name'].unique()
        
        if len(results) > 0:
            print(f"   ✅ 找到 {len(results)} 个相关节点。最匹配的前 10 个如下：")
            sorted_results = sorted(results, key=len)
            for i, res in enumerate(sorted_results[:10]):
                print(f"      [{i+1}] {res}")
        else:
            print(f"   ❌ 完全没有找到包含 '{keyword}' 的节点。")

    print("\n==================================================")
    print("💡 检索完毕。请复制最准确的名称并替换到 Step 2 中。")
    print("==================================================")

if __name__ == "__main__":
    precise_search_missed_nodes()